# DQ 检查 — ODS / DWD 层（Rules 1-12）

Rules 1-10: ODS 完整性/范围/时序
Rules 11-12: DWD 行数守恒 + 参照完整性

任何 FAIL 将抛出异常，阻断 DWS 下游任务。

## 1. SparkSession

In [1]:
import os
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("TLC-DQ-AfterDWD")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE       = os.path.expanduser("~/tlc-spark-pipeline")
ODS_Y      = f"{BASE}/data/warehouse/ods/yellow"
ODS_G      = f"{BASE}/data/warehouse/ods/green"
DWD_FACT   = f"{BASE}/data/warehouse/dwd/fact_trips"
DWD_ZONE   = f"{BASE}/data/warehouse/dwd/dim_zone"
QUARANTINE = f"{BASE}/data/warehouse/dq/quarantine"

TARGET_MONTH = os.environ.get("TARGET_MONTH", "2024-03")
yr, mo = int(TARGET_MONTH.split("-")[0]), int(TARGET_MONTH.split("-")[1])
print(f"TARGET_MONTH={TARGET_MONTH}")

print(f"Spark {spark.version} ready | TARGET_MONTH={TARGET_MONTH}")


## 2. DQ 框架

In [2]:
results = []

def check(rule_id, layer, rule_type, description, passed, bad_count=0, total=0, threshold=None):
    status = "PASS" if passed else "FAIL"
    results.append({
        "rule_id": rule_id, "layer": layer, "rule_type": rule_type,
        "description": description, "status": status,
        "bad_count": bad_count, "total": total, "threshold": threshold
    })
    mark = "✅" if passed else "❌"
    print(f"{mark} Rule {rule_id:02d} [{layer}] {description}  bad={bad_count:,}")


## 3. Rules 1-10：ODS 层检查（Yellow + Green）

In [3]:
ods_y = spark.read.parquet(ODS_Y).filter(
    (F.year("pickup_datetime") == yr) & (F.month("pickup_datetime") == mo))
ods_g = spark.read.parquet(ODS_G).filter(
    (F.year("pickup_datetime") == yr) & (F.month("pickup_datetime") == mo))
ods   = ods_y.unionByName(ods_g, allowMissingColumns=True)
ods_total = ods.count()
print(f"ODS (yellow+green) total: {ods_total:,}")

# Rules 1-5: 完整性
# Rules 1-4: 关键字段严格不允许 null
for rule_id, field in enumerate(
    ["fare_amount", "pickup_datetime", "PULocationID", "DOLocationID"],
    start=1
):
    bad = ods.filter(col(field).isNull()).count()
    check(rule_id, "ODS", "completeness", f"{field} not null",
          passed=(bad == 0), bad_count=bad, total=ods_total)

# Rule 5: passenger_count 允许最多 15% null（TLC 上游已知问题，DWD 层过滤）
bad5 = ods.filter(col("passenger_count").isNull()).count()
check(5, "ODS", "completeness", "passenger_count not null (threshold 15%)",
      passed=(bad5 / ods_total < 0.15), bad_count=bad5, total=ods_total, threshold=0.15)

# Rule 6: fare_amount > 0
bad6 = ods.filter(col("fare_amount") <= 0).count()
check(6, "ODS", "range", "fare_amount > 0", passed=(bad6 / ods_total < 0.05),
      bad_count=bad6, total=ods_total, threshold=0.05)

# Rule 7: trip_distance >= 0
bad7 = ods.filter(col("trip_distance") < 0).count()
check(7, "ODS", "range", "trip_distance >= 0", passed=(bad7 == 0),
      bad_count=bad7, total=ods_total)

# Rule 8: passenger_count in [1,6]
bad8 = ods.filter(~col("passenger_count").between(1, 6)).count()
check(8, "ODS", "range", "passenger_count in [1,6]", passed=(bad8 / ods_total < 0.15),
      bad_count=bad8, total=ods_total, threshold=0.15)

# Rule 9: dropoff > pickup
bad9 = ods.filter(col("dropoff_datetime") <= col("pickup_datetime")).count()
check(9, "ODS", "temporal", "dropoff > pickup", passed=(bad9 / ods_total < 0.01),
      bad_count=bad9, total=ods_total)

# Rule 10: 时间在 2024 Q1
bad10 = ods.filter(
    (col("pickup_datetime") < "2024-01-01") | (col("pickup_datetime") >= "2024-04-01")
).count()
check(10, "ODS", "temporal", "pickup in 2024-Q1", passed=(bad10 / ods_total < 0.01),
      bad_count=bad10, total=ods_total)


## 4. Rules 11-12：DWD 层检查

In [4]:
fact       = spark.read.parquet(DWD_FACT).filter(
    (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo))
fact_count = fact.count()
dim_zone   = spark.read.parquet(DWD_ZONE)
print(f"fact_trips: {fact_count:,}")

# Rule 11: 行数守恒（含 FHV HV，ODS total 需加上 fhv）
from pyspark.sql import functions as F
ods_fhv_total = spark.read.parquet(
    f"{BASE}/data/warehouse/ods/fhv_hv").count()
ods_all_total = ods_total + ods_fhv_total
retention = fact_count / ods_all_total
check(11, "DWD", "row_conservation", "fact >= ODS_all × 0.80",
      passed=(retention >= 0.80), bad_count=ods_all_total - fact_count, total=ods_all_total)

# Rule 12: 参照完整性 PULocationID ∈ dim_zone
valid_zones = set(r.location_id for r in dim_zone.collect())
bad12 = fact.filter(~col("PULocationID").isin(list(valid_zones))).count()
check(12, "DWD", "referential", "PULocationID in dim_zone",
      passed=(bad12 == 0), bad_count=bad12, total=fact_count)


## 5. 违规记录写入隔离表

In [5]:
quarantine_records = (
    ods
    .withColumn("dq_reason",
        F.when(col("fare_amount").isNull(),         "null_fare_amount")
        .when(col("pickup_datetime").isNull(),       "null_pickup_datetime")
        .when(col("PULocationID").isNull(),          "null_PULocationID")
        .when(col("DOLocationID").isNull(),          "null_DOLocationID")
        .when(col("passenger_count").isNull(),       "null_passenger_count")
        .when(col("fare_amount") <= 0,               "non_positive_fare")
        .when(col("trip_distance") < 0,              "negative_distance")
        .when(~col("passenger_count").between(1,6),  "invalid_passenger_count")
        .when(col("dropoff_datetime") <= col("pickup_datetime"), "invalid_time_order")
        .when(
            (col("pickup_datetime") < "2024-01-01") |
            (col("pickup_datetime") >= "2024-04-01"), "out_of_range_datetime")
    )
    .filter(col("dq_reason").isNotNull())
    .withColumn("check_ts", F.current_timestamp())
)

q_count = quarantine_records.count()
q_rate  = q_count / ods_total
print(f"Quarantine records: {q_count:,} / {ods_total:,} = {q_rate:.2%}")
quarantine_records.groupBy("dq_reason").count().orderBy(F.desc("count")).show()
quarantine_records.write.mode("overwrite").partitionBy("source_type").parquet(QUARANTINE)
print(f"Quarantine written to: {QUARANTINE}")


## 6. DQ 报告 + 阻断判断

In [6]:
fail_rules = [r for r in results if r['status'] == 'FAIL']
pass_rules = [r for r in results if r['status'] == 'PASS']

print(f"\n{'='*50}")
print(f"DQ Report (ODS+DWD)  {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Rules: {len(results)} total | {len(pass_rules)} PASS | {len(fail_rules)} FAIL")
print(f"Quarantine rate: {q_rate:.2%}")
print(f"={'='*49}")

spark.stop()

if fail_rules:
    print("FAIL rules (blocking):")
    for r in fail_rules:
        print(f"  ❌ Rule {r['rule_id']:02d}: {r['description']}  bad={r['bad_count']:,}")
    raise ValueError(
        f"DQ FAILED: {len(fail_rules)} blocking rule(s). "
        "Pipeline halted — fix upstream data before proceeding to DWS."
    )
else:
    print("All ODS/DWD rules passed — proceeding to DWS.")

print("DQ after-DWD check complete.")
